# Bacteria-ID 两阶段训练

按“环境 → 数据检查 → ref 训练/验证 → finetune 训练/验证 → 测试评估 → 打包”执行。
两阶段共享 30 类标签；finetune 从 ref 的 model.pt 初始化，BID 不进入主数据集注册表。

## 1. 环境与数据包

In [ ]:
"""环境引导：定位项目根目录，按需 pip 安装缺失依赖。"""
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
INSTALL_MISSING_DEPS = True
PIP_INDEX_URL = os.environ.get("PIP_INDEX_URL") or "https://pypi.tuna.tsinghua.edu.cn/simple"
PACKAGE_ARCHIVE_NAME = "ramanv2.zip"
DATA_ARCHIVE_NAME = "data.npz"
REQUIRED_MODULES = {
    "torch": "torch", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "tqdm": "tqdm", "pandas": "pandas", "seaborn": "seaborn",
    "yaml": "PyYAML", "openpyxl": "openpyxl",
}

def _has_payload(directory: Path) -> bool:
    return (directory / PACKAGE_ARCHIVE_NAME).is_file() or (directory / "ramanv2" / "core" / "config.py").is_file()

override = PROJECT_ROOT_OVERRIDE or os.environ.get("RAMAN_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((c for c in (cwd, *cwd.parents) if _has_payload(c)), cwd)

missing = [pkg for mod, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(mod) is None]
if missing and INSTALL_MISSING_DEPS:
    command = [sys.executable, "-m", "pip", "install", "--no-cache-dir"]
    if PIP_INDEX_URL:
        command += ["--index-url", PIP_INDEX_URL]
    subprocess.run(command + missing, shell=False, check=True)
print(f"project_root = {PROJECT_ROOT}")


In [ ]:
# 输入检查：代码包必需；data.npz 可选，dataset/<数据集>/ 已在位时可跳过恢复。
REQUIRED_ARCHIVES = {"ramanv2.zip", DATA_ARCHIVE_NAME}
absent_archives = sorted(
    name for name in REQUIRED_ARCHIVES if not (PROJECT_ROOT / name).is_file()
)
if PACKAGE_ARCHIVE_NAME in absent_archives and not (PROJECT_ROOT / "ramanv2").is_dir():
    raise FileNotFoundError(
        f"缺少代码包：请把 {PACKAGE_ARCHIVE_NAME} 或解压后的 ramanv2/ 放到 {PROJECT_ROOT}"
    )
if DATA_ARCHIVE_NAME in absent_archives:
    print(f"未找到 {DATA_ARCHIVE_NAME}：若 dataset/<数据集>/init 已在位可忽略。")
print("输入检查完成。")


In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

PACKAGE_ARCHIVE = PROJECT_ROOT / PACKAGE_ARCHIVE_NAME
PACKAGE_DIR = PROJECT_ROOT / "ramanv2"


def load_package():
    """检测到压缩包时替换 ramanv2；否则使用已解压的包。"""
    if PACKAGE_ARCHIVE.is_file():
        temp_dir = Path(tempfile.mkdtemp(prefix="ramanv2_unpack_", dir=PROJECT_ROOT))
        try:
            with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
                archive.extractall(temp_dir)
            unpacked_dir = temp_dir / PACKAGE_DIR.name
            if not unpacked_dir.is_dir():
                raise FileNotFoundError("压缩文件中未找到 ramanv2/ 目录")
            if PACKAGE_DIR.exists():
                if not PACKAGE_DIR.is_dir():
                    raise FileExistsError(f"包路径不是目录：{PACKAGE_DIR}")
                shutil.rmtree(PACKAGE_DIR)
            shutil.move(str(unpacked_dir), str(PROJECT_ROOT))
            print(f"replaced package: {PACKAGE_ARCHIVE.name} -> {PACKAGE_DIR}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
    elif not PACKAGE_DIR.is_dir():
        raise FileNotFoundError(f"请先将 {PACKAGE_ARCHIVE_NAME} 放到 {PROJECT_ROOT}")
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))


load_package()


In [ ]:
import importlib
import sys


def reload_ramanv2():
    """清除 ramanv2 模块缓存，以加载当前目录中的最新代码。"""
    for name in list(sys.modules):
        if name == "ramanv2" or name.startswith("ramanv2."):
            del sys.modules[name]
    importlib.invalidate_caches()
    config_module = importlib.import_module("ramanv2.core.config")
    print("ramanv2 modules reloaded.")
    return config_module


config_module = reload_ramanv2()
build_config = config_module.build_config


## 2. 恢复 BID 数据

In [ ]:
from ramanv2.packaging.platform_bundle import restore_data_archive

# BID 数据包放在 dataset/bacteriaID/data.npz；三份目录已存在时可关闭。
RESTORE_DATA_ENABLE = False
BID_DATA_ARCHIVE = PROJECT_ROOT / "dataset" / "bacteriaID" / "data.npz"

if RESTORE_DATA_ENABLE and BID_DATA_ARCHIVE.is_file():
    restore_report = restore_data_archive(BID_DATA_ARCHIVE, PROJECT_ROOT)
    print(
        f"restored {BID_DATA_ARCHIVE.name}: samples={restore_report['sample_count']} "
        f"stages={restore_report['sample_counts']}"
    )
elif RESTORE_DATA_ENABLE:
    print(f"未找到 {BID_DATA_ARCHIVE}，使用已有 dataset/bacteriaID 目录")
else:
    print("已跳过 BID 数据恢复。")

## 3. BID 数据与配置


In [ ]:
from datetime import datetime

from ramanv2.bid.config import build_bid_config
from ramanv2.bid.evaluation import evaluate_bid_model_run, evaluate_bid_test_set
from ramanv2.bid.paths import build_bid_selection, resolve_bid_paths
from ramanv2.common.paths import resolve_path
from ramanv2.data.count import count_dataset

# 三份数据已按类目扁平存放，BID 不参与数据集注册表和 UI 操作。
SEED = 42
# 两阶段分别设置 batch_size；None 使用默认 64。
REF_BATCH_SIZE = 256
FINETUNE_BATCH_SIZE = 64
CPU_ENABLE = False
# 实验根：None 自动生成 output/bacteriaID/<时间戳>；填写已有运行根可只重跑某一阶段。
EXPERIMENT_ROOT = None
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = resolve_path(EXPERIMENT_ROOT) or (
    PROJECT_ROOT / "output" / "bacteriaID" / RUN_TAG
)
# 复用已有第一阶段时，直接指向其槽位；ref 训练格会写入同一个值。
ref_slot_dir = OUTPUT_ROOT / f"ref_{SEED}" / "level_1"
print(f"ref_slot_dir = {ref_slot_dir}  model_exists={(ref_slot_dir / 'model.pt').is_file()}")

paths = resolve_bid_paths(PROJECT_ROOT)
selection = build_bid_selection(paths)
for label, directory in (
    ("train_ref", paths.train_ref_dir),
    ("train_finetune", paths.train_finetune_dir),
    ("test", paths.test_dir),
):
    if not directory.is_dir():
        raise FileNotFoundError(f"缺少 BID 数据目录：{directory}")
    class_count = sum(1 for item in directory.iterdir() if item.is_dir())
    _, sample_count = count_dataset(directory)
    print(f"{label:15} classes={class_count} samples={sample_count}  dir={directory}")

## 4. 第一阶段 ref 训练


In [ ]:
from ramanv2.training.workflow import TrainRequest, run_training

ref_values = {"seed": SEED}
if REF_BATCH_SIZE is not None:
    ref_values["batch_size"] = int(REF_BATCH_SIZE)
ref_config = build_bid_config("ref", ref_values)
ref_experiment_dir = OUTPUT_ROOT / f"ref_{SEED}"

run_training(
    TrainRequest(
        config=ref_config,
        level_name="level_1",
        experiment_dir=ref_experiment_dir,
        train_dir=paths.train_ref_dir,
        selection=selection,
        train_archive_enable=False,
    )
)
ref_slot_dir = ref_experiment_dir / "level_1"
print(f"ref_slot_dir = {ref_slot_dir}")

## 5. 第一阶段验证


In [ ]:
import json

from IPython.display import Image, display

ref_val_dir = evaluate_bid_model_run(
    ref_slot_dir,
    "level_1",
    "cpu" if CPU_ENABLE else None,
    train_dir=paths.train_ref_dir,
)
ref_summary = json.loads((ref_val_dir / "metrics.json").read_text(encoding="utf-8"))["summary"]
print(
    f"ref val accuracy={ref_summary['accuracy'] * 100:.4f}%  "
    f"macro_f1={ref_summary['macro_f1'] * 100:.4f}%  support={ref_summary['support']}"
)
display(Image(filename=str(ref_val_dir / "confusion_matrix.png")))


## 6. 第二阶段 finetune 训练


In [ ]:
from ramanv2.bid.finetune import build_checkpoint_initializer
from ramanv2.training.workflow import TrainRequest, run_training

# 只重跑微调时，前面的 ref 训练格可以跳过；ref_slot_dir 已在配置格给出。
# 微调专用目录前缀；epoch 使用 BID 默认配置。
FINETUNE_RUN_NAME = "finetune"
finetune_values = {"seed": SEED}
if FINETUNE_BATCH_SIZE is not None:
    finetune_values["batch_size"] = int(FINETUNE_BATCH_SIZE)
finetune_config = build_bid_config("finetune", finetune_values)
finetune_experiment_dir = OUTPUT_ROOT / f"{FINETUNE_RUN_NAME}_{SEED}"
initialize_from_ref = build_checkpoint_initializer(ref_slot_dir / "model.pt")

run_training(
    TrainRequest(
        config=finetune_config,
        level_name="level_1",
        experiment_dir=finetune_experiment_dir,
        train_dir=paths.train_finetune_dir,
        selection=selection,
        train_archive_enable=False,
        initialize_model=initialize_from_ref,
    )
)
finetune_slot_dir = finetune_experiment_dir / "level_1"
print(f"finetune_slot_dir = {finetune_slot_dir}")

## 7. 第二阶段验证


In [ ]:
import json

from IPython.display import Image, display

# 只重跑微调时，本格不依赖前面 ref 验证格的 import。
finetune_val_dir = evaluate_bid_model_run(
    finetune_slot_dir,
    "level_1",
    "cpu" if CPU_ENABLE else None,
    train_dir=paths.train_finetune_dir,
)
finetune_summary = json.loads((finetune_val_dir / "metrics.json").read_text(encoding="utf-8"))["summary"]
print(
    f"finetune val accuracy={finetune_summary['accuracy'] * 100:.4f}%  "
    f"macro_f1={finetune_summary['macro_f1'] * 100:.4f}%  support={finetune_summary['support']}"
)
display(Image(filename=str(finetune_val_dir / "confusion_matrix.png")))

## 8. 独立测试集


In [ ]:
from IPython.display import Image, display
from ramanv2.evaluation.slot_flow import format_test_report

# 是否额外评估 ref 模型；通常只看 finetune，设为 False 可节省时间。
REF_TEST_ENABLE = False

test_reports = {}
test_targets = []
if REF_TEST_ENABLE:
    test_targets.append(("ref", ref_slot_dir, paths.train_ref_dir))
test_targets.append(("finetune", finetune_slot_dir, paths.train_finetune_dir))

for stage, slot_dir, train_dir in test_targets:
    report = evaluate_bid_test_set(
        slot_dir,
        "cpu" if CPU_ENABLE else None,
        paths.test_dir,
        None,
        train_dir=train_dir,
    )
    test_reports[stage] = report
    print()
    print(f"===== {stage} =====")
    print(format_test_report(report))
    display(Image(filename=str(Path(report["result_dir"]) / "confusion_matrix.png")))


## 9. 打包实验结果


In [ ]:
from ramanv2.packaging.experiment_archive import publish_experiment_archive

# 打包整个运行根（ref/ 与 finetune/ 及其结果），压缩包落在项目根。
PACKAGE_RESULT_ENABLE = True
archive_path = publish_experiment_archive(
    OUTPUT_ROOT.resolve(),
    package_enable=PACKAGE_RESULT_ENABLE,
)
print(f"archive = {archive_path}")